# Trợ lý tài chính đa phương thức (portfolio)

Bạn dựng một trợ lý tài chính cá nhân chạy local: **ảnh hoá đơn** thành các khoản chi (vision LLM), **phân loại** chi tiêu,
**câu hỏi bằng giọng nói** thành câu trả lời có số (Whisper, LLM hiểu ý định, code tính số, TTS đọc lại), và một **agent
báo cáo tháng** không được phép bịa số. Dữ liệu cá nhân là **giả lập**: 24 hoá đơn `cord-v2` gán ngày và cửa hàng tự đặt,
câu hỏi giọng nói do chính TTS đọc.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`. Test kiểm các hàm này trên dữ liệu nhỏ, không gọi model.
- Phần còn lại (pipeline thật, đo, báo cáo) là của bạn: thêm cell `explore` tuỳ ý (dòng đầu `# explore: ...`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/18-multimodal/exercises/advanced/01-finance-assistant -q`

In [ ]:
import json
import re
from dataclasses import dataclass

CATEGORIES = ["ăn uống", "đồ uống", "mua sắm", "đi lại", "giải trí", "khác"]
INTENTS = ["total", "by_category", "top_item", "compare", "other"]


@dataclass
class Expense:
    date: str        # "YYYY-MM-DD"
    merchant: str
    item: str
    amount: float    # đồng
    category: str = "khác"

## 1. Từ JSON hoá đơn tới sổ chi tiêu

- `parse_amount(text)`: dán từ bài Basic 02.
- `expenses_from_receipt(pred, date, merchant) -> list[Expense]`: `pred` là JSON model đọc được (dạng của bài
  Intermediate 01) hoặc `None`. Mỗi phần tử dict của `items` có `amount` đọc được và **dương** thành một `Expense` (tên là
  `name` đã `strip`, rỗng thì `"(không tên)"`). Không có dòng nào dùng được mà `total` đọc được và dương thì trả **một**
  khoản `"(tổng hoá đơn)"` bằng tổng. `None` trả `[]`.
- `categorize(name, rules, llm) -> str`: `rules` là dict `nhóm -> list từ khoá`. Chuẩn hoá tên (chữ hoa, ký tự không phải chữ
  cái hay chữ số thành dấu cách); nếu một từ khoá xuất hiện như **một từ trọn vẹn** thì trả nhóm đầu tiên (theo thứ tự dict,
  rồi thứ tự từ khoá) chứa nó, **không gọi LLM**. Ngược lại gọi `llm.chat(messages, json_schema=...)` **đúng một lần**
  với schema `{"category": enum CATEGORIES}`; JSON hỏng, nhóm ngoài `CATEGORIES`, hoặc lời gọi lỗi thì `"khác"`.
- `monthly_summary(expenses, month) -> dict` với `month` dạng `"YYYY-MM"`: `month`, `total`, `by_category` (dict nhóm →
  tổng, sắp giảm dần theo tiền, hoà thì theo tên nhóm), `n_receipts` (số cặp `(date, merchant)` khác nhau), `top_items` (tối
  đa 3 cặp `(item, amount)` lớn nhất, hoà thì theo ngày rồi tên món).

In [ ]:
def parse_amount(text) -> float | None:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def expenses_from_receipt(pred: dict | None, date: str, merchant: str) -> list[Expense]:
    """Các dòng hàng có số tiền dương; không có thì một khoản bằng tổng hoá đơn."""
    # TODO
    raise NotImplementedError


def categorize(name: str, rules: dict[str, list[str]], llm) -> str:
    """Luật từ khoá (từ trọn vẹn) trước; không khớp thì gọi LLM một lần với JSON schema enum."""
    # TODO
    raise NotImplementedError


def monthly_summary(expenses: list[Expense], month: str) -> dict:
    """{"month", "total", "by_category", "n_receipts", "top_items"} của một tháng "YYYY-MM"."""
    # TODO
    raise NotImplementedError

## 2. Câu hỏi bằng giọng nói: PII, ý định, câu trả lời do code tính

- `mask_pii(text) -> str`: thay email bằng `[EMAIL]`; chuỗi 13 tới 19 chữ số (cho phép dấu cách hoặc gạch giữa các chữ số)
  **qua kiểm tra Luhn** bằng `[CARD]` (không qua thì giữ nguyên); số điện thoại Việt Nam (`0` hoặc `+84` rồi đúng 9 chữ số,
  cho phép dấu cách hoặc dấu chấm xen giữa) bằng `[PHONE]`. Áp theo thứ tự email, thẻ, điện thoại. Xem module 19 mục 3.
- `route(question, llm) -> dict`: gọi LLM **một lần** với JSON schema, trả `{"intent", "month", "category"}`: `intent` thuộc
  `INTENTS`, `month` là chuỗi `"YYYY-MM"` hợp lệ (tháng 01 tới 12) hoặc `None`, `category` thuộc `CATEGORIES` hoặc `None`.
  Giá trị không hợp lệ của `month`/`category` thành `None`; JSON hỏng, intent lạ, hoặc lời gọi lỗi thì
  `{"intent": "other", "month": None, "category": None}`.
- `format_vnd(x) -> str`: làm tròn tới đồng, dấu chấm phân cách nghìn, thêm `" đồng"`: `1250000` thành `"1.250.000 đồng"`.

In [ ]:
def mask_pii(text: str) -> str:
    """Che email, số thẻ (qua Luhn), số điện thoại Việt Nam."""
    # TODO
    raise NotImplementedError


def route(question: str, llm) -> dict:
    """LLM -> {"intent", "month", "category"} đã kiểm; lỗi thì intent "other"."""
    # TODO
    raise NotImplementedError


def format_vnd(x: float) -> str:
    """1250000 -> "1.250.000 đồng"."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: pipeline của bạn: 24 hoá đơn -> sổ chi tiêu -> phân loại -> câu hỏi giọng nói -> báo cáo tháng
# Gợi ý cấu trúc (xem README): cache mọi lời gọi model ra đĩa; đo từng khối; ghi bảng vào REPORT.md